# Run the method: requirements → constraints → architecture

> 📓 *Companion to* **Modern Agentic AI Engineer** · Ch 42 §42.1–42.2, §42.4–42.6, §42.8 · type: worksheet

**The promise:** by the end you will have run the chapter's full seven-step method on a system *you* pick, and you'll leave with a populated design doc — ranked NFRs, an estimate, two or three weighed candidates, a failure/degradation table, a data plan, and at least one ADR with rejected alternatives.

This is a *worksheet*, not a code lab. It runs fully offline and free — no API key, no network, just the standard library and a little arithmetic. You'll edit fill-in cells and let a few tiny cells assemble them into a real artifact you can drop into `templates/system-design-doc/`.

## 🧠 Why this matters

Reading the method and running it are different skills, and only the second one transfers to a roadmap meeting or a Ch 52 interview. The whole point of §42.1 is that an architecture is the *residue of constraints*: pin down what must be true and how well, then what reality permits, and the viable shapes collapse from dozens to two or three — with the choice between them an explicit, recordable trade-off.

The worksheet's ordering does one structural thing for you: it **forbids drawing boxes first**. You cannot fill the architecture cell until you've ranked the NFRs and run the estimate above it. That single constraint is what separates a defensible design from a diagram you reverse-justified. If, by the end, many shapes still look equally fine, you haven't found the binding constraint yet — and the worksheet will tell you so.

## Objectives & prereqs

**By the end you can:**
- Take a vague ask to a defensible, *recorded* architecture in about an hour.
- Produce ranked NFRs with the three AI-specific ones (quality, cost/task, safety) pinned numerically.
- Build a per-dependency failure table and a degradation ladder.
- Write an ADR with rejected alternatives, modeled on the chapter's ADR-031.
- Emit the §42.8 playbook skeleton filled in for your system.

**Prereqs:** [`42-01-back-of-envelope-estimation.ipynb`](42-01-back-of-envelope-estimation.ipynb) — you'll embed its estimate here. Chapter 27 (quality attributes, ADRs, C4) and Chapter 29 (the failure laws) read.

**Packages:** standard library only (`dataclasses`, `textwrap`). Nothing beyond `requirements.txt`; no API key; fully offline.

In [ ]:
# Setup — imports, env, and the MOCK switch.
import os
import textwrap
from dataclasses import dataclass, field

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; never hardcode keys

# This worksheet is pure local reflection + arithmetic — it makes no model calls in
# EITHER mode — but we read COMPANION_MOCK so the habit and message stay consistent
# across the whole course (see docs/HOW-TO-USE.md, "the MOCK covenant").
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

print("System-design worksheet — offline, free, no API key needed.")
print(f"COMPANION_MOCK = {os.getenv('COMPANION_MOCK', '1')} (this notebook makes no model calls either way)")
print()
print("Throughout, the *worked example* lines come from §42.7 (the agentic support system).")
print("Replace each ✍️ cell with YOUR system; the worked answers are there to imitate.")

## 🧠 The seven steps (the method you're about to run)

The method runs from *what must be true* to *what to build*, and every later step may push back on an earlier one (§42.1). The cells below walk these in order — you cannot honestly fill step 4 before steps 1–3.

1. **Clarify functional requirements** — what it does, for whom, what's out of scope.
2. **Pin the NFRs, ranked** — SLOs, scale, latency, cost, quality, safety (Ch 27).
3. **Estimate** — traffic, tokens, dollars, storage (reuse 42-01). *Numbers kill bad designs.*
4. **Sketch the architecture** — components + data flow; two or three candidates, not one.
5. **Design for failure and scale** — timeouts, retries, breakers, fallbacks; the degradation ladder; back-pressure (Ch 29).
6. **Design the data architecture** — serving plane + exhaust plane (the trace→eval flywheel).
7. **Decide, record, define success** — the ADR with rejected alternatives, and the metrics that prove it works.

> 🎯 Keep asking, at every step: **“at the expense of what?”** If you finish without asking it at least three times, you ran the steps but skipped the method (§42.1).

## ✍️ Step 0 — Pick your system

Name the system you'll design for the rest of the worksheet. Pick something real — a feature on your roadmap, or the chapter's prompt (“an AI assistant for our customers”). Set `SYSTEM_NAME` and write the one-paragraph ask, including **explicit non-goals** (§42.8 step 1).

In [ ]:
# ✍️ EDIT ME.
SYSTEM_NAME = "Agentic support system"   # <- name your system

PROBLEM_STATEMENT = textwrap.dedent("""\
    Automate inbound customer support for a mid-size B2B SaaS company. Handle
    tickets from chat and email; answer from product docs and the customer's
    account data; resolve what it can, draft for humans otherwise. It may issue
    refunds up to $100 WITH human approval.
    Non-goals: phone support; anything beyond the refund tool that touches billing.
""").strip()

print(SYSTEM_NAME)
print("=" * len(SYSTEM_NAME))
print(PROBLEM_STATEMENT)

## Step 1 — Clarify: the questions that eliminate designs

Refuse to design until the requirements are concrete (§42.2). “An AI assistant for support” is not a requirement; “deflect 40% of tier-1 tickets at under \$0.30 per ticket without ever losing a ticket” is. Answer each clarifying question for *your* system — every answer eliminates designs.

In [ ]:
# ✍️ EDIT ME. Answer each clarifying question (§42.2) for YOUR system.
CLARIFY = {
    "Who uses it, and how many at peak?": "Support agents + end customers; ~10k tickets/day.",
    "Interactive or background?":          "Both: chat is interactive (stream); email is background.",
    "Latency SLO, exactly?":               "Chat first token < 3 s; email tolerates minutes.",
    "What does an error cost?":            "A wrong refund is money; a wrong answer escalates, not apologizes.",
    "Cost budget per task?":               "<= $0.30 per ticket against a $6 human cost.",
    "What must NEVER happen?":             "Lose a ticket; ungated refund; cross-tenant data leakage.",
}

print(f"Clarifying questions for: {SYSTEM_NAME}\n" + "-" * 64)
for q, a in CLARIFY.items():
    print(f"  Q: {q}")
    print(f"  A: {a}\n")

if any(not a.strip() for a in CLARIFY.values()):
    print("⚠️  Some answers are blank — a blank answer is an un-eliminated design. Fill them.")

## Step 2 — Pin the NFRs, ranked

For AI systems, add three first-class NFRs to the classic availability/latency/throughput list and **pin them with numbers**: *quality* (a measurable eval target), *cost per task*, and *safety posture* (§42.2 key idea). Then **rank** them — ranking is the whole game, because the top-ranked requirement is the one that gets to veto the architecture.

Note where different parts of the system deserve *different* SLOs (intake vs. resolution): that asymmetry is a gift — it lets you make a tiny piece extremely reliable instead of making everything expensive.

In [ ]:
@dataclass
class NFR:
    name: str
    target: str                 # PIN IT WITH A NUMBER
    applies_to: str = "whole system"  # note where parts deserve different SLOs


# ✍️ EDIT ME. Rank top-to-bottom: rank 1 gets to VETO the architecture.
# The three AI-specific NFRs (quality, cost/task, safety) MUST be pinned numerically.
RANKED_NFRS = [
    NFR("Never lose a ticket", "99.95% intake success", applies_to="intake (crown jewel)"),
    NFR("Safety posture",      "refunds always human-gated; per-customer isolation"),
    NFR("Cost per task",       "<= $0.30 / ticket (vs $6 human cost)"),
    NFR("Quality",             ">= 90% eval pass on resolved tickets; wrong answers escalate"),
    NFR("Latency",             "chat first token < 3 s", applies_to="resolution (email: minutes)"),
    NFR("Availability of resolution", "ranks BELOW all of the above — deliberately"),
]

print(f"Ranked NFRs for: {SYSTEM_NAME}\n" + "-" * 64)
for i, nfr in enumerate(RANKED_NFRS, 1):
    print(f"  {i}. {nfr.name:32s} {nfr.target}")
    if nfr.applies_to != "whole system":
        print(f"     ↳ applies to: {nfr.applies_to}")

ai_specific = {"quality", "cost per task", "safety posture"}
pinned = {n.name.lower() for n in RANKED_NFRS}
missing = ai_specific - pinned
print()
print("✓ all three AI-specific NFRs pinned." if not missing
      else f"⚠️  pin these AI-specific NFRs with numbers: {sorted(missing)}")

## Step 3 — Estimate (reuse 42-01)

Drop in the back-of-envelope estimate for *your* assumptions and let the numbers eliminate options before you draw anything. We inline a trimmed copy of 42-01's `estimate()` so this worksheet stands alone; in your own work, import it or re-run that notebook. The §42.7 worked support case is shown — replace the assumptions with yours.

In [ ]:
SECONDS_PER_DAY = 86_400
PRICE_INPUT_PER_M, PRICE_OUTPUT_PER_M = 1.00, 5.00  # round illustrative prices (§42.3)


def quick_estimate(tasks_per_day, in_tok_per_task, out_tok_per_task, peak_factor=8.0):
    """Trimmed twin of 42-01's estimate(): tokens and dollars per task/day.

    Pass the *per-task totals* (already summed across the agent's loop) — the
    way §42.7 quotes them (≈ 9,000 input / 1,500 output PER TICKET, not per call).
    Don't re-multiply by loop depth here, or you triple-count the bill.
    """
    in_per_day = tasks_per_day * in_tok_per_task
    out_per_day = tasks_per_day * out_tok_per_task
    cost_day = in_per_day / 1e6 * PRICE_INPUT_PER_M + out_per_day / 1e6 * PRICE_OUTPUT_PER_M
    return {
        "avg_per_s": tasks_per_day / SECONDS_PER_DAY,
        "peak_per_s": tasks_per_day / SECONDS_PER_DAY * peak_factor,
        "cost_day": cost_day,
        "cost_task": cost_day / tasks_per_day if tasks_per_day else 0.0,
    }


# ✍️ EDIT ME — the §42.7 worked support-system assumptions; swap for yours.
# in_tok/out_tok are PER-TICKET TOTALS (≈ 3 model calls already summed in).
est = quick_estimate(tasks_per_day=10_000, in_tok_per_task=9_000, out_tok_per_task=1_500)

print(f"traffic : {est['avg_per_s']:.2f}/s avg, {est['peak_per_s']:.1f}/s peak — trivial")
print(f"cost    : ${est['cost_day']:,.0f}/day ≈ ${est['cost_task']:.3f}/task")
print()
print("vs §42.7 Step 3: ≈ $165/day, ≈ $0.017/task — way under the $0.30 budget.")
print("CONCLUSION the numbers force: this design is shaped by RELIABILITY and COST,")
print("not throughput. Don't penny-pinch models; spend the headroom on QUALITY.")

## Step 4 — Sketch: two or three candidates, weighed against the ranked NFRs

Now — and *only* now, with requirements and numbers in hand — sketch the shape. Always two or three candidates, never one, each scored against your ranked NFRs (§42.1, §42.8 step 5). The top-ranked NFR is the veto: in the worked case, “never lose a ticket” forces intake and resolution to be *different services*, because one combined service would chain the crown-jewel SLO to its flakiest dependency.

In [ ]:
# ✍️ EDIT ME. List 2–3 candidate shapes and how each fares on your TOP-ranked NFR.
CANDIDATES = [
    {
        "name": "A. Single combined service (intake + resolution together)",
        "verdict": "REJECTED — chains 99.95% intake SLO to flaky model/tool deps.",
    },
    {
        "name": "B. Split: tiny stateless intake API + async agent worker pool",
        "verdict": "CHOSEN — intake only validates/persists/enqueues; resolution can\n"
                   "             degrade without losing a ticket. Scales on queue depth.",
    },
    {
        "name": "C. Fully managed agent platform (buy, don't build)",
        "verdict": "REJECTED — cedes the gating + eval control that safety (rank #2) requires.",
    },
]

top = RANKED_NFRS[0]
print(f"Candidates weighed against the TOP-ranked NFR: '{top.name}' ({top.target})\n" + "-" * 64)
for c in CANDIDATES:
    print(f"  {c['name']}")
    print(f"     -> {c['verdict']}\n")

if len(CANDIDATES) < 2:
    print("⚠️  You sketched one shape. The method requires 2–3 — list the ones you rejected and why.")

## Step 5 — Failure design: per-dependency table + degradation ladder

For every arrow in your diagram, ask: what happens when the thing at the end of it is slow, down, or answering garbage (§42.4, Ch 29)? Fill the table — timeout / retry / breaker / fallback per dependency — remembering the AI-specific tuning: **cost-aware retries** (a retried model call re-spends real money), **per-provider breakers**, and a ladder that ends in *honesty*.

In [ ]:
@dataclass
class Dependency:
    name: str
    timeout: str
    retry: str
    breaker: str
    fallback: str


# ✍️ EDIT ME. One row per dependency your agent calls.
DEPENDENCIES = [
    Dependency("Model provider", "30 s/call + whole-run deadline", "2 + jitter (capped low)",
               "per-provider", "gateway -> second provider"),
    Dependency("Retrieval (RAG)", "5 s", "1", "per-store", "answer without retrieved context, flagged"),
    Dependency("Reply tool (send)", "10 s", "2 (idempotency key per ticket-step)", "per-tool",
               "queue for later; never double-send"),
    Dependency("Refund tool", "n/a — human-gated", "none (irreversible)", "per-tool", "route to human approval"),
]

print(f"Per-dependency failure design for: {SYSTEM_NAME}\n" + "=" * 78)
print(f"{'dependency':18s} {'timeout':22s} {'retry':24s} {'fallback'}")
print("-" * 78)
for d in DEPENDENCIES:
    print(f"{d.name:18s} {d.timeout:22s} {d.retry:24s} {d.fallback}")

### 🔮 Predict your weakest arrow

Before you write the degradation ladder, **predict**: which dependency above is your *weakest arrow* — the one most likely to be the first domino in an outage? Write it down. Then write the ladder, and check whether your ladder actually protects against that arrow failing. (In the worked case the weakest arrow is the model provider; the ladder's whole job is to make a provider brownout survivable.)

In [ ]:
# ✍️ EDIT ME — your prediction, then the ladder.
WEAKEST_ARROW_PREDICTION = "Model provider — everything routes through it."

# The degradation ladder: ordered list of what you serve as conditions worsen (§42.4).
# It MUST end in honesty, not a crash.
DEGRADATION_LADDER = [
    "full agent (retrieve + tools + draft/send)",
    "smaller / cheaper model",
    "retrieval-only suggested answer",
    "queued: 'a human will reply'",
    "honest error — but the ticket is already safe in Postgres",
]

print(f"🔮 predicted weakest arrow: {WEAKEST_ARROW_PREDICTION}\n")
print("Degradation ladder (worsening conditions ->):")
for i, rung in enumerate(DEGRADATION_LADDER, 1):
    print(f"  {i}. {rung}")

last = DEGRADATION_LADDER[-1].lower()
print()
print("✓ ladder ends in honesty." if ("error" in last or "human" in last or "safe" in last)
      else "⚠️  the last rung should be an honest error/queue, not a crash.")

## Step 6 — Data: two planes, a freshness SLO, and the flywheel

Every agentic system carries two data planes, and conflating them is a design smell (§42.6). The **serving plane** is what the agent reads (RAG corpus, account data); its central commitment is a **freshness SLO** — pick the *cheapest* pipeline that meets it (nightly batch is enough far more often than teams assume). The **exhaust plane** is what the system emits (traces, costs, outcomes) — design the trace→eval **flywheel** in from day one, as boxes on the diagram, not an ops afterthought.

In [ ]:
# ✍️ EDIT ME.
SERVING_PLANE = {
    "sources": "product docs; customer account data (CRM, read-only)",
    "freshness_SLO": "docs: 24 h (nightly re-ingest); resolved tickets join weekly after review",
    "cheapest_pipeline_that_meets_it": "nightly batch re-sync + re-embed changed docs; idempotent, re-runnable",
}

EXHAUST_PLANE = {
    "emits": "full trace per run (cost + outcome), token counts, feedback ratings",
    "flywheel": "escalations + bad ratings -> eval cases -> evals gate every prompt/tool/model change in CI",
}

print("SERVING PLANE (what the agent reads):")
for k, v in SERVING_PLANE.items():
    print(f"  {k:34s}: {v}")
print("\nEXHAUST PLANE (what the system emits):")
for k, v in EXHAUST_PLANE.items():
    print(f"  {k:34s}: {v}")

print("\nThe flywheel, as boxes (not an afterthought):")
print("  run -> trace store -> harvest failures/feedback -> eval set -> CI gate -> better quality -> more usage -> (loop)")

## Step 7 — Record: an ADR with rejected alternatives

Write an ADR for your single most expensive decision, *including rejected alternatives* — the chapter's ADR-031 is the worked model to imitate (§42.7, Ch 27). An ADR without rejected options isn't a decision record; it's a press release. The cell below assembles a real ADR string from your fill-ins.

In [ ]:
# ✍️ EDIT ME — your single most expensive decision, modeled on ADR-031.
ADR = {
    "id": "ADR-001",
    "title": "Separate ticket intake from agent resolution",
    "status": "Accepted — 2026-06-20",
    "context": "'Never lose a ticket' (99.95%) outranks every other requirement. Agent "
               "resolution depends on model providers, retrieval, and tools — none of which "
               "can honestly promise 99.95%. One combined service chains the crown-jewel SLO "
               "to its flakiest dependency.",
    "decision": "Split the system: a minimal stateless intake API (validate, persist, enqueue "
                "— no model or tool calls) and a separate worker pool that resolves tickets "
                "from the queue, scaled on queue depth.",
    "consequences_pos": "Intake's failure surface is tiny; resolution can degrade or pause "
                        "without losing tickets; the two scale and deploy independently.",
    "consequences_neg": "Two deployables and a queue to operate; replies become asynchronous, "
                        "so chat needs streaming/polling UX.",
    "rejected": "single service with aggressive retries (chains the SLOs); fully managed agent "
                "platform (cedes the gating and eval control that safety rank #2 requires).",
}


def render_adr(adr: dict) -> str:
    return textwrap.dedent(f"""\
        # {adr['id']}: {adr['title']}

        ## Status
        {adr['status']}

        ## Context
        {textwrap.fill(adr['context'], 72)}

        ## Decision
        {textwrap.fill(adr['decision'], 72)}

        ## Consequences
        + {textwrap.fill(adr['consequences_pos'], 70)}
        - {textwrap.fill(adr['consequences_neg'], 70)}
        - Rejected: {textwrap.fill(adr['rejected'], 62)}
        """)


adr_text = render_adr(ADR)
print(adr_text)
assert "rejected" in adr_text.lower(), "An ADR without rejected alternatives is not a decision record."

## ⚠️ Pitfall: drawing boxes first, discovering requirements later

The most common way a design goes wrong — and the one this worksheet structurally forbids — is **drawing the architecture first and reverse-justifying the requirements** to fit it. You feel productive (boxes! arrows!) while skipping the only step that makes the boxes defensible. The tell is an architecture that no ranked NFR actually *forced*: if you removed your top requirement and the diagram wouldn't change, the diagram was never driven by it.

**The fix** is the ordering you just followed: clarify → rank → estimate → *then* sketch. The cell below runs a cheap check — it confirms your top NFR is the one your chosen candidate cites. If the chosen shape doesn't trace back to the #1 requirement, you drew boxes first.

In [ ]:
chosen = next((c for c in CANDIDATES if "CHOSEN" in c["verdict"].upper()), None)
top_nfr = RANKED_NFRS[0]

print(f"Top-ranked NFR : {top_nfr.name}")
print(f"Chosen shape   : {chosen['name'] if chosen else '(none marked CHOSEN!)'}")
print()
if chosen is None:
    print("⚠️  No candidate is marked CHOSEN — you haven't decided. Mark one and re-run.")
else:
    # Did the rejection of the OTHER candidates cite the top NFR's concern?
    rationale = " ".join(c["verdict"] for c in CANDIDATES).lower()
    driver_words = top_nfr.name.lower().split() + top_nfr.target.lower().split()
    traced = any(w in rationale for w in driver_words if len(w) > 3)
    print("✓ the chosen shape traces back to the #1 requirement — requirements drove the boxes."
          if traced else
          "⚠️  your rationale never mentions the #1 requirement — did you draw boxes first?")

## 🎯 Senior lens

The method's output is not a diagram; it is *surfaced trade-offs* (§42.1). Every step exists to drag an expensive decision into the open while changing it still costs a conversation instead of a quarter. The junior deliverable from a design session is a clean architecture diagram; the senior deliverable is a short list of the trade-offs you consciously made and what you gave up for each.

So the real test of this worksheet isn't whether the cells are filled — it's whether you asked **“at the expense of what?”** at least three times. Splitting intake from resolution buys reliability *at the expense of* two deployables and async UX. Spending the cost headroom on a stronger model buys quality *at the expense of* margin. Nightly batch buys simplicity *at the expense of* freshness. If your filled-in doc records those costs explicitly, you ran the method. If it only records the wins, you ran the steps and skipped the method.

## 📋 The §42.8 playbook, filled in for your system

Everything above compresses into the reusable playbook — the same artifact serves design docs, reviews, and interviews. The cell assembles your fill-ins into the §42.8 skeleton; copy its output into [`templates/system-design-doc/`](../../../templates/system-design-doc/) as the start of a real design doc, and run the closing checklist.

In [ ]:
def render_design_doc() -> str:
    nfr_lines = "\n".join(f"     {i}. {n.name} — {n.target}" for i, n in enumerate(RANKED_NFRS, 1))
    ladder = " -> ".join(DEGRADATION_LADDER)
    return textwrap.dedent(f"""\
        # Design: {SYSTEM_NAME}            Date: 2026-06-20 / Author: you / Status: draft

        1. Problem & users — {textwrap.shorten(PROBLEM_STATEMENT, 200)}
        2. Functional requirements — see clarifying answers ({len(CLARIFY)} questions resolved).
        3. NFRs, RANKED:
        {nfr_lines}
        4. Estimates — ${est['cost_day']:,.0f}/day, ${est['cost_task']:.3f}/task, {est['peak_per_s']:.1f}/s peak.
        5. Architecture — chosen: {chosen['name'] if chosen else 'TBD'} (of {len(CANDIDATES)} weighed).
        6. Failure design — {len(DEPENDENCIES)} dependencies covered; ladder: {ladder}
        7. Data — serving freshness SLO: {SERVING_PLANE['freshness_SLO']}; flywheel: traces -> evals -> CI gate.
        8. Decisions — {ADR['id']}: {ADR['title']} (with rejected alternatives).
        9. Success metrics & rollout — TODO: name what you measure and how you launch.
        """)


print(render_design_doc())

In [ ]:
# The §42.8 readiness checklist, run against what you filled in above.
checks = {
    "Scope + non-goals written down":            "Non-goals" in PROBLEM_STATEMENT,
    "Quality/cost/safety pinned AND ranked":     len(RANKED_NFRS) >= 3,
    "Tokens + dollars computed, not guessed":    est["cost_day"] > 0,
    "2–3 candidates weighed against NFRs":       len(CANDIDATES) >= 2,
    "Every dependency: timeout/retry/breaker/fallback": all(
        d.timeout and d.retry and d.breaker and d.fallback for d in DEPENDENCIES),
    "Degradation ladder defined, ends honestly": len(DEGRADATION_LADDER) >= 3,
    "Freshness SLO stated":                       bool(SERVING_PLANE["freshness_SLO"]),
    "Trace->eval flywheel in the design":         "flywheel" in EXHAUST_PLANE,
    "Irreversible action human-gated":           any("human" in d.fallback.lower() for d in DEPENDENCIES),
    "Expensive decision recorded as an ADR":     "rejected" in ADR and bool(ADR["rejected"]),
}

print("📋 §42.8 readiness checklist\n" + "-" * 50)
for item, ok in checks.items():
    print(f"  [{'x' if ok else ' '}] {item}")
passed = sum(checks.values())
print(f"\n{passed}/{len(checks)} checks pass.", "Ship the doc." if passed == len(checks)
      else "Fill the gaps above before calling the design defensible.")

## Recap

- An architecture is the **residue of constraints**: clarify → rank NFRs → estimate → *then* sketch.
- Pin the three AI-specific NFRs (**quality, cost/task, safety**) with numbers, and **rank** them — rank 1 vetoes.
- The estimate (from 42-01) **eliminates options** before you draw; here it said “shaped by reliability + cost.”
- Failure design is **per dependency** (timeout/retry/breaker/fallback) plus a **degradation ladder that ends in honesty**.
- Two data planes: a **freshness SLO** picks the cheapest serving pipeline; the **trace→eval flywheel** is on the diagram.
- Record the expensive decision as an **ADR with rejected alternatives**, and the output is **surfaced trade-offs**, not a diagram.

## Exercises

Each exercise *changes* something and asks you to predict the result before running.

1. **Re-rank and watch the shape move.** Swap NFR #1 and #3 (make *cost* outrank *never lose a ticket*). Predict how the chosen candidate should change, then rewrite `CANDIDATES`. Does a single combined service become defensible?
2. **Halve the loop depth.** A shorter agent loop roughly halves the per-ticket tokens — re-run Step 3 with `in_tok_per_task` and `out_tok_per_task` cut in half. Predict the new \$/task and what it does to the p95 latency story before you run it. Where does the saved budget go (§42.7's “spend headroom on quality”)?
3. **A different system.** Re-do Steps 0–2 for a *background* system (e.g. nightly report generation). Predict which NFR becomes #1 when nothing is interactive — does latency fall off the list?
4. **Stress the ladder.** Add a “provider total outage” row to `DEPENDENCIES` and predict which rung of your degradation ladder catches it. If none does, your ladder has a hole — fix it.

In [ ]:
# Exercise scratch space — your code here.


In [ ]:
# Exercise scratch space — your code here.


## Next

- **Template:** drop your filled-in playbook into [`templates/system-design-doc/`](../../../templates/system-design-doc/) — 42-02's output is the canonical instance of that template; the ADR shape feeds [`templates/adr-template/`](../../../templates/adr-template/).
- **Book:** §42.8 (the reusable playbook + 📋 checklist) and §42.7 (the worked support system this imitates); forward to Ch 43, which applies the method to four named architectures, and Ch 52, which reuses it as the interview drill.
- **Where this leads:** the patterns you specified are realized in [`blueprints/observability-stack/`](../../../blueprints/observability-stack/) (the flywheel) and [`blueprints/agent-loop/`](../../../blueprints/agent-loop/) (reliability seams); your worked support answer matches the `capstone/`'s intake/resolution split and returns as the readiness checklist in Ch 44.